### Function calling in Open AI

In [ ]:
from openai import OpenAI

client = OpenAI()
tools = [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get current temperature for a given location.",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "City and country e.g. Bogotá, Colombia"
                }
            },
            "required": [
                "location"
            ],
            "additionalProperties": False
        },
        "strict": True
    }
}]

completion = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": "What is the weather like in Paris today?"}],
    tools=tools
)

print(completion.choices[0].message.tool_calls)

### Function calling in LangChain

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain.prompts import ChatPromptTemplate
from langchain.tools import tool

# Define a tool with defined schema.
@tool
def get_weather(location: str, day: str) -> str:
    """Return real-time temperature, humidity, and conditions for the given location and day """
    pass

tools = [get_weather]
llm = ChatOpenAI(model="gpt-4o", temperature=0.5)

SYSTEM_PROMPT = """You are a helpful weather assistant.
If the user asks about weather, decide whether to call the `get_weather` tool.
Only call tools when needed; otherwise, answer directly."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{input}")
])

# Create a tool-calling agent and an executor.
agent = create_tool_calling_agent(llm, tools, prompt)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# Run: the agent decides if/when to call the tool and with what args.
user_q = "What is the weather like in Paris today?"
result = executor.invoke({"input": user_q})
print("\n=== Final Answer ===")
print(result["output"])

In [ ]:
[
  {
    "id": "call_12345xyz",
    "type": "function",
    "function": {
      "name": "get_weather",
      "arguments": "{\"location\":\"Paris, France\"}"
    }
  }
]